# 실습 4. 세 군 이상의 크기 비교

사회약학 연구방법 노트의 '실습 4. 세 군 이상의 크기 비교'에 딸린 노트북입니다. MASS 패키지의 birthwt 자료(출생아와 산모 189명)로 어머니 인종 분류에 따른 세 군의 출생체중을 분산분석, Welch 분산분석, Tukey 사후분석, Kruskal-Wallis 검정과 Dunn 검정으로 비교하고, 어머니 체중 삼분위에 Jonckheere-Terpstra 검정을 직접 만들어 적용합니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 4에 있습니다.


## a. 실습 데이터 준비

이 실습이 쓰는 패키지를 불러와 버전을 확인한 뒤 birthwt 자료를 읽습니다. 자료는 인터넷에서 읽어 오므로 인터넷 연결이 필요합니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import scipy, statsmodels, matplotlib, seaborn

print("pandas     ", pd.__version__)
print("numpy      ", np.__version__)
print("scipy      ", scipy.__version__)
print("statsmodels", statsmodels.__version__)
print("matplotlib ", matplotlib.__version__)
print("seaborn    ", seaborn.__version__)

### 셀 2. 데이터 불러오기와 군 이름표

In [ ]:
url = ("https://vincentarelbundock.github.io/Rdatasets/"
       "csv/MASS/birthwt.csv")
df = pd.read_csv(url)
df["race_f"] = pd.Categorical(
    df["race"].map({1: "White", 2: "Black", 3: "Other"}),
    categories=["White", "Black", "Other"])
df["race_f"].value_counts(sort=False)

### 셀 3. 군별 요약통계

In [ ]:
g = df.groupby("race_f", observed=True)["bwt"]
g.agg(["count", "mean", "std", "median"]).round(1)

### 셀 4. 상자그림과 개별 점

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x="race_f", y="bwt", ax=ax,
            color="0.85", width=0.5, showfliers=False)
np.random.seed(0)        # 점 흩뿌리기(jitter) 위치 고정
sns.stripplot(data=df, x="race_f", y="bwt", ax=ax,
              color="black", alpha=0.4, jitter=0.2)
ax.set_xlabel("Maternal race")
ax.set_ylabel("Birth weight (g)")
plt.show()

### 셀 5. 군별 흡연 비율

In [ ]:
pd.crosstab(df["race_f"], df["smoke"], normalize="index").round(2)

## b. 분산분석과 사후분석

### 셀 6. 일원배치 분산분석 (scipy)

In [ ]:
white = df.loc[df["race_f"] == "White", "bwt"]
black = df.loc[df["race_f"] == "Black", "bwt"]
other = df.loc[df["race_f"] == "Other", "bwt"]

res = stats.f_oneway(white, black, other)
print(res)
print(f"F = {res.statistic:.3f}, p = {res.pvalue:.4f}")

### 셀 7. 분산분석표 (statsmodels)

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

model = smf.ols("bwt ~ race_f", data=df).fit()
table = sm.stats.anova_lm(model)
table

### 셀 8. 효과크기 η²와 ω²

In [ ]:
ss_b = table.loc["race_f", "sum_sq"]      # 군간 제곱합
ss_w = table.loc["Residual", "sum_sq"]    # 군내 제곱합
ms_w = table.loc["Residual", "mean_sq"]   # 군내 평균제곱
eta2 = ss_b / (ss_b + ss_w)
omega2 = (ss_b - 2 * ms_w) / (ss_b + ss_w + ms_w)
print(f"SS_B = {ss_b:,.0f}, SS_W = {ss_w:,.0f}")
print(f"eta^2 = {eta2:.3f}, omega^2 = {omega2:.3f}")
print(f"pooled SD = {np.sqrt(ms_w):.1f} g")

### 셀 9. 잔차의 정규성

In [ ]:
resid = model.resid                 # 각 아기의 값 - 자기 군 평균
sw = stats.shapiro(resid)
print(f"Shapiro-Wilk W = {sw.statistic:.3f}, p = {sw.pvalue:.3f}")

fig, ax = plt.subplots(figsize=(4.2, 3.6))
stats.probplot(resid, dist="norm", plot=ax)   # 잔차 189개의 정규 Q-Q plot
ax.set_title("Normal Q-Q plot of residuals")
plt.show()

### 셀 10. 등분산 확인과 Welch 분산분석

In [ ]:
from statsmodels.stats.oneway import anova_oneway

lev = stats.levene(white, black, other)       # 중앙값 기준
print(f"Levene: p = {lev.pvalue:.3f}")
welch = anova_oneway(df["bwt"], df["race_f"], use_var="unequal")
print(f"Welch F = {welch.statistic:.3f}, "
      f"df = ({welch.df_num:.0f}, {welch.df_denom:.1f}), "
      f"p = {welch.pvalue:.4f}")

### 셀 11. Tukey HSD 사후분석

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(df["bwt"], df["race_f"], alpha=0.05)
print(tukey)

## c. Kruskal-Wallis 검정

### 셀 12. Kruskal-Wallis 검정

In [ ]:
kw = stats.kruskal(white, black, other)
print(f"H = {kw.statistic:.3f}, p = {kw.pvalue:.4f}")

df["rank"] = df["bwt"].rank()      # 189명 전체에서 매긴 순위
print(df.groupby("race_f", observed=True)["rank"].mean().round(1))

### 셀 13. 쌍별 Mann-Whitney 검정과 Holm 보정

In [ ]:
from statsmodels.stats.multitest import multipletests

groups = {"White": white, "Black": black, "Other": other}
pairs = [("White", "Black"), ("White", "Other"),
         ("Black", "Other")]
p_mw, sup = [], []
for a, b in pairs:
    r = stats.mannwhitneyu(groups[b], groups[a])
    p_mw.append(r.pvalue)
    sup.append(r.statistic / (len(groups[a]) * len(groups[b])))
p_mw_holm = multipletests(p_mw, method="holm")[1]
print("P(b > a):", np.round(sup, 3))
print("p:", np.round(p_mw, 4), " Holm:", np.round(p_mw_holm, 4))

### 셀 14. Dunn 검정 직접 계산하기

In [ ]:
N = len(df)
t = df["bwt"].value_counts()               # 동점 묶음의 크기
tie = (t**3 - t).sum() / (12 * (N - 1))    # 동점 보정 항
mean_rank = df.groupby("race_f", observed=True)["rank"].mean()
n = df["race_f"].value_counts()

p_dunn = []
for a, b in pairs:
    se = np.sqrt((N * (N + 1) / 12 - tie) * (1 / n[a] + 1 / n[b]))
    z = (mean_rank[a] - mean_rank[b]) / se
    p_dunn.append(2 * stats.norm.sf(abs(z)))
p_dunn_holm = multipletests(p_dunn, method="holm")[1]
print(np.round(p_dunn, 4), np.round(p_dunn_holm, 4))

## d. Jonckheere-Terpstra 검정

### 셀 15. 어머니 체중 삼분위 만들기

In [ ]:
# 사전 가설: 어머니 체중(lwt)이 무거울수록 출생체중이 크다
df["lwt3"] = pd.qcut(df["lwt"], q=3, labels=["T1", "T2", "T3"])
df.groupby("lwt3", observed=True).agg(
    n=("bwt", "size"), lwt_min=("lwt", "min"),
    lwt_max=("lwt", "max"), bwt_median=("bwt", "median"))

### 셀 16. J 통계량 계산 함수

In [ ]:
def jt_J(groups):
    """J = 모든 (앞 군, 뒤 군) 쌍에서 뒤 군 값이 더 큰 짝의 수"""
    J = 0.0
    for i in range(len(groups)):
        for j in range(i + 1, len(groups)):
            a = np.asarray(groups[i])[:, None]   # 앞 군 (세로)
            b = np.asarray(groups[j])[None, :]   # 뒤 군 (가로)
            J += (b > a).sum() + 0.5 * (b == a).sum()
    return J

order = ["T1", "T2", "T3"]
tert = [df.loc[df["lwt3"] == k, "bwt"] for k in order]
print("J =", jt_J(tert))

### 셀 17. J는 Mann-Whitney U의 합

In [ ]:
U12 = stats.mannwhitneyu(tert[1], tert[0]).statistic   # T2 > T1
U13 = stats.mannwhitneyu(tert[2], tert[0]).statistic   # T3 > T1
U23 = stats.mannwhitneyu(tert[2], tert[1]).statistic   # T3 > T2
print(U12, U13, U23, U12 + U13 + U23)

### 셀 18. 정규근사로 p-value 구하기

In [ ]:
def jt_test(groups):
    """Jonckheere-Terpstra 검정: 동점 보정 정규근사, 양측 p"""
    J = jt_J(groups)
    n = np.array([len(x) for x in groups])      # 군별 인원
    N = n.sum()
    allv = np.concatenate([np.asarray(x) for x in groups])
    t = np.unique(allv, return_counts=True)[1]  # 동점 묶음 크기
    EJ = (N**2 - (n**2).sum()) / 4               # 귀무가설 아래 평균
    v1 = (N*(N-1)*(2*N+5) - (n*(n-1)*(2*n+5)).sum()
          - (t*(t-1)*(2*t+5)).sum()) / 72
    v2 = ((n*(n-1)*(n-2)).sum() * (t*(t-1)*(t-2)).sum()
          / (36 * N * (N-1) * (N-2)))
    v3 = (n*(n-1)).sum() * (t*(t-1)).sum() / (8 * N * (N-1))
    z = (J - EJ) / np.sqrt(v1 + v2 + v3)
    return J, EJ, z, 2 * stats.norm.sf(abs(z))

J, EJ, z, p = jt_test(tert)
print(f"J = {J:.0f}, E(J) = {EJ:.0f}, z = {z:.2f}, p = {p:.4f}")

statsmodels가 0.15보다 낮으면 이 셀에서 `ImportError`가 납니다. 그때는 이 셀을 건너뛰고 다음 셀부터 실행하세요. 다음 셀들은 직접 만든 `jt_test`를 씁니다.

### 셀 19. statsmodels 0.15의 함수와 비교

In [ ]:
import statsmodels
print("statsmodels", statsmodels.__version__)
from statsmodels.stats.nonparametric import jonckheere_terpstra

res_sm = jonckheere_terpstra(tert, alternative="two-sided")
print(f"J = {res_sm.statistic:.0f}, z = {res_sm.zstat:.2f}, "
      f"p = {res_sm.pvalue:.4f}")

### 셀 20. 순열검정으로 확인하기

In [ ]:
rng = np.random.default_rng(2026)      # 난수 시드 고정
y = df["bwt"].to_numpy()
lab = df["lwt3"].to_numpy()
B = 10000
J_perm = np.empty(B)
for r in range(B):
    s = rng.permutation(lab)           # 군 이름표를 무작위로 섞음
    J_perm[r] = jt_J([y[s == k] for k in order])
extreme = np.abs(J_perm - EJ) >= abs(J - EJ)   # 관측값만큼 극단적
print("extreme:", extreme.sum(), "of", B)
print(f"permutation p = {(extreme.sum() + 1) / (B + 1):.4f}")

### 셀 21. 같은 삼분위에 Kruskal-Wallis 검정

In [ ]:
kw3 = stats.kruskal(*tert)
print(f"H = {kw3.statistic:.2f}, p = {kw3.pvalue:.4f}")

### 셀 22. 순서대로 커지지 않는 경우

In [ ]:
df["ftv3"] = pd.cut(df["ftv"], bins=[-1, 0, 1, 99],
                    labels=["0", "1", "2+"])
visits = [df.loc[df["ftv3"] == k, "bwt"] for k in ["0", "1", "2+"]]
g3 = df.groupby("ftv3", observed=True)["bwt"]
print(g3.agg(["count", "median"]))
J, EJ, z, p = jt_test(visits)
print(f"JT: z = {z:.2f}, p = {p:.3f}")
print(f"KW: p = {stats.kruskal(*visits).pvalue:.3f}")

## 과제 정답

**과제 1.** 셀 11의 Tukey 결과에서 `Other  White 297.4352  0.026` 행을 한 문장으로 옮기고, 이 결과가 말해 주지 않는 것을 두 가지 이상 쓰세요. 셀 7의 모형에 흡연(smoke)을 더해 White와 Other의 차이가 어떻게 바뀌는지 확인하세요.

### 셀 23. 과제 1 정답. Tukey 결과의 한 행 읽기

In [ ]:
m_smoke = smf.ols("bwt ~ race_f + smoke", data=df).fit()   # 흡연을 함께 넣음
print("adjusted 95% CI, Other:")
print(m_smoke.conf_int().loc["race_f[T.Other]"].round(1).tolist())
pd.DataFrame({"crude": model.params,
              "adjusted": m_smoke.params}).round(1)

**과제 2.** 연구 질문이 처음부터 "White 산모와 비교해 나머지 두 군의 출생체중이 다른가"였다고 하고, 셀 11의 Tukey 대신 Dunnett 방법(`stats.dunnett`, 대조군 White)으로 다시 비교하세요. Tukey와 비교해 P와 신뢰구간이 어떻게 바뀌는지, 왜 그런지, 그리고 Dunnett 결과로 말할 수 없는 것이 무엇인지 설명하세요.

### 셀 24. 과제 2 정답. Tukey 대신 Dunnett 방법으로

In [ ]:
dun = stats.dunnett(black, other, control=white, rng=2026)   # 대조군 = White
ci_d = dun.confidence_interval()
for k, name in enumerate(["Black", "Other"]):      # 넣은 순서대로
    print(f"{name} - White: p = {dun.pvalue[k]:.3f}")
    print(f"  95% CI {ci_d.low[k]:.1f} ~ {ci_d.high[k]:.1f}")

**과제 3.** 동료가 인종 분류의 세 군을 출생체중 중앙값이 작은 순서(Other, Black, White)로 늘어놓고 셀 18의 `jt_test`로 Jonckheere-Terpstra 검정을 한 뒤 "출생체중은 Other, Black, White 순서로 커지는 추세가 있었다(z = 2.54, P for trend = 0.011)"라고 썼습니다. 동료의 분석을 재현하고, 인종과 출생체중이 무관한 자료를 여러 번 만들어 같은 절차를 되풀이하면 '유의'가 몇 %나 나오는지 확인한 뒤, 무엇이 왜 틀렸는지 설명하세요.

### 셀 25. 과제 3 정답. 자료를 본 뒤 정한 순서로 한 추세 검정

In [ ]:
med = df.groupby("race_f", observed=True)["bwt"].median()
order_seen = med.sort_values().index.tolist()     # 자료를 본 뒤 정한 순서
print("medians:", order_seen)
print("means  :", g.mean().sort_values().index.tolist())
J_c, EJ_c, z_c, p_c = jt_test(
    [df.loc[df["race_f"] == k, "bwt"] for k in order_seen])
print(f"z = {z_c:.2f}, P for trend = {p_c:.3f}")

rng3 = np.random.default_rng(2026)
y3, race_lab = df["bwt"].to_numpy(), df["race_f"].to_numpy()
B3, hit_jt, hit_kw = 2000, 0, 0
for _ in range(B3):
    s = rng3.permutation(race_lab)         # 인종과 무관하게 섞음
    gs = sorted([y3[s == k] for k in ["White", "Black", "Other"]],
                key=np.median)             # 매번 중앙값 순서로 늘어놓음
    hit_jt += jt_test(gs)[3] < 0.05
    hit_kw += stats.kruskal(*gs).pvalue < 0.05
print(f"'significant' in {B3} null data sets:")
print(f"  JT, order from data {hit_jt / B3:.3f}")
print(f"  Kruskal-Wallis      {hit_kw / B3:.3f}")